# 01 — Ground truth các thay đổi giữa các phiên bản MITRE ATT&CK

**Đề tài:** Trôi phiên bản tri thức trong trợ lý CTI dùng LLM/RAG
**Mục tiêu của notebook:** xác định **chính xác, tất định** những gì đã thay đổi giữa các bản phát hành ATT&CK Enterprise, phân loại theo kiểu thay đổi, và xuất ra các bảng mà mọi phần sau của đề tài đều dùng tới.

| Đầu ra | Dùng cho |
|---|---|
| `change_events.csv`: mọi sự kiện thay đổi giữa hai phiên bản liên tiếp | Số liệu động cơ cho bài báo; Trụ 1 (đo độ trễ cập nhật) |
| `summary_by_transition.csv`: số sự kiện theo từng lần chuyển phiên bản × kiểu thay đổi | Bảng/biểu đồ trong mục *Motivation* |
| `id_resolution_latest.csv`: mọi mã từng tồn tại → trạng thái và mã thay thế ở bản mới nhất | Baseline "tra bảng ánh xạ" của Trụ 3; chấm điểm câu trả lời có mã cũ |
| `changed_items_18.1_to_latest.csv`: danh sách thực thể đã đổi giữa v18.1 và bản mới nhất | Sinh câu hỏi trước/sau cho Trụ 2 |

## Phân loại các kiểu thay đổi

Notebook phát hiện các kiểu thay đổi sau, **chỉ từ dữ liệu STIX chính thức**:

| Mã | Kiểu thay đổi | Cách phát hiện |
|---|---|---|
| `ADDED` | Kỹ thuật mới | Có ở bản sau, không có ở bản trước |
| `REVOKED→NEW` | Thu hồi, chuyển sang mã **mới tạo** (đổi mã) | Quan hệ `revoked-by`, đích chưa tồn tại ở bản trước |
| `REVOKED→EXISTING` | Thu hồi, **gộp vào** một kỹ thuật đã có | Quan hệ `revoked-by`, đích đã tồn tại ở bản trước |
| `MERGE` | Nhiều mã cùng bị thu hồi về **một** đích | ≥ 2 sự kiện thu hồi cùng đích trong một lần chuyển |
| `DEPRECATED` | Ngừng dùng, **không có mã thay thế** | `x_mitre_deprecated = true` |
| `RENAMED` | Đổi tên, giữ mã | Tên khác nhau, cùng mã, cùng còn hiệu lực |
| `TACTIC_CHANGED` | Đổi tactic mà kỹ thuật thuộc về | Tập tactic khác nhau |
| `REPARENTED` | Đổi kỹ thuật cha | Quan hệ `subtechnique-of` khác nhau |
| `MITIGATION_CHANGED` | Thêm/bớt biện pháp giảm thiểu | Tập quan hệ `mitigates` khác nhau |
| `MAJOR_REVISION` | Nội dung sửa lớn, giữ mã | Phần nguyên của `x_mitre_version` tăng (*ứng viên* đổi nghĩa) |
| `REFINED` | Kỹ thuật được bổ sung sub-technique mới | Có kỹ thuật con mới |
| `TACTIC_REDEFINED` | Tactic giữ mã nhưng đổi tên/nghĩa | Cùng mã TA, khác `x_mitre_shortname` (ca TA0005 ở v19) |

**Giới hạn cần nói rõ trong bài:** một lần *tách* (split, 1 → N) **không suy ra được trọn vẹn** từ STIX, vì quan hệ `revoked-by` chỉ trỏ tới **một** đích. Notebook ghi nhận được các dấu hiệu (kỹ thuật được bổ sung sub-technique, kỹ thuật bị ngừng dùng). Danh sách tách đầy đủ phải đối chiếu thêm *release notes* của MITRE. Chính giới hạn này là lý do cách "tra bảng ánh xạ" không đủ, và là chỗ đứng cho phương pháp ở Trụ 3.

## 1. Cấu hình

- `VERSIONS`: các phiên bản đưa vào phân tích, mặc định là **bản cuối của mỗi phiên bản lớn** từ 6.3 đến 19.2, cộng thêm **19.0** để đối chiếu với release notes chính thức của v19.
- Đặt `USE_ALL_VERSIONS = True` để phân tích đủ 41 bản. Khi đó cần khoảng 1,5 GB đĩa và lâu hơn.

In [1]:
from pathlib import Path
import json, hashlib, datetime, collections
import requests
import pandas as pd

ROOT = Path.cwd()
RAW  = ROOT / "data" / "raw" / "attack_versions"
OUT  = ROOT / "drift"
RAW.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

VERSIONS = ["6.3", "7.2", "8.2", "9.0", "10.1", "11.3", "12.1", "13.1",
            "14.1", "15.1", "16.1", "17.1", "18.1", "19.0", "19.2"]
USE_ALL_VERSIONS = False
INDEX_URL = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/index.json"
pd.set_option("display.max_colwidth", 80)

## 2. Tải các phiên bản và ghi mã băm

Danh sách phiên bản và đường dẫn lấy từ `index.json` chính thức của MITRE (`mitre-attack/attack-stix-data`). Mỗi file được lưu lại kèm SHA-256 trong `drift/manifest.json` để tái lập.

In [2]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

index = requests.get(INDEX_URL, timeout=60).json()
ent = next(c for c in index["collections"] if c["name"] == "Enterprise ATT&CK")
release = {v["version"]: (v["url"], v["modified"][:10]) for v in ent["versions"]}

def vkey(v):
    return tuple(int(x) for x in v.split("."))

if USE_ALL_VERSIONS:
    VERSIONS = sorted(release, key=vkey)
VERSIONS = sorted(VERSIONS, key=vkey)
missing = [v for v in VERSIONS if v not in release]
assert not missing, f"Không có trong index: {missing}"

manifest = {"created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "index_url": INDEX_URL, "versions": {}}
for v in VERSIONS:
    url, date = release[v]
    p = RAW / f"enterprise-attack-{v}.json"
    if not p.exists():
        r = requests.get(url, timeout=300); r.raise_for_status(); p.write_bytes(r.content)
    manifest["versions"][v] = {"url": url, "release_date": date, "sha256": sha256(p)}
    print(f"v{v:5s} {date}  {p.stat().st_size/1e6:5.1f} MB  {manifest['versions'][v]['sha256'][:12]}…")
_ = (OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))

v6.3   2020-03-09   14.1 MB  91a073f92d1f…
v7.2   2020-07-15   19.7 MB  04728e4b995e…
v8.2   2021-01-27   22.3 MB  8af8ba82d52c…
v9.0   2021-04-29   25.9 MB  a41aea2db10e…
v10.1  2021-11-10   31.2 MB  0a999035f26f…
v11.3  2022-07-07   35.1 MB  4bb7a2bc2624…
v12.1  2022-11-08   38.2 MB  e84679af4bc4…
v13.1  2023-05-09   40.2 MB  5ec90131dc59…
v14.1  2023-11-14   41.8 MB  13af7514ad1b…
v15.1  2024-05-02   43.5 MB  a57988bffe40…
v16.1  2024-11-12   46.5 MB  8423d8dac3fc…
v17.1  2025-05-06   44.9 MB  0d1c347a4d58…
v18.1  2025-11-13   50.7 MB  f857d8f78f2f…
v19.0  2026-04-28   52.7 MB  df520ea0775a…
v19.2  2026-08-05   53.8 MB  dc1639caa550…


## 3. Đọc mỗi phiên bản thành một "ảnh chụp" chuẩn hoá

Mỗi phiên bản được rút gọn thành các bảng cần thiết: kỹ thuật (mã, tên, trạng thái, tactic, kỹ thuật cha, `x_mitre_version`), tactic, biện pháp giảm thiểu, quan hệ `mitigates`, và quan hệ `revoked-by`.

**Trạng thái** của một đối tượng: `active` (còn hiệu lực), `revoked` (bị thu hồi, có mã thay thế) hoặc `deprecated` (ngừng dùng, không có mã thay thế). Đối tượng bị thu hồi **vẫn nằm trong file** của các bản sau. Nhờ vậy ta truy vết được mã cũ.

In [3]:
def ext_id(o):
    for r in o.get("external_references", []):
        if r.get("source_name") == "mitre-attack" and r.get("external_id"):
            return r["external_id"]

def status(o):
    if o.get("revoked"):
        return "revoked"
    if o.get("x_mitre_deprecated"):
        return "deprecated"
    return "active"

def load_snapshot(v):
    objs = json.loads((RAW / f"enterprise-attack-{v}.json").read_text())["objects"]
    s2i, tactics, tech, mit = {}, {}, {}, {}
    for o in objs:
        i = ext_id(o)
        if i:
            s2i[o["id"]] = i
        if o["type"] == "x-mitre-tactic" and status(o) == "active":
            tactics[o["x_mitre_shortname"]] = (i, o["name"])
    ta_by_id = {tid: (sn, name) for sn, (tid, name) in tactics.items()}
    for o in objs:
        i = ext_id(o)
        if not i:
            continue
        if o["type"] == "attack-pattern":
            tac = sorted({tactics[k["phase_name"]][0] for k in o.get("kill_chain_phases", [])
                          if k.get("kill_chain_name") == "mitre-attack" and k["phase_name"] in tactics})
            tech[i] = dict(stix=o["id"], name=o["name"], status=status(o), tactics=tuple(tac),
                           is_sub=bool(o.get("x_mitre_is_subtechnique")), parent=None,
                           version=o.get("x_mitre_version", "1.0"))
        elif o["type"] == "course-of-action":
            mit[i] = dict(name=o["name"], status=status(o))
    revoked_by, mitigates = {}, set()
    for o in objs:
        if o["type"] != "relationship":
            continue
        src, dst, rt = s2i.get(o["source_ref"]), s2i.get(o["target_ref"]), o["relationship_type"]
        if not src or not dst:
            continue
        if rt == "revoked-by":
            revoked_by[src] = dst
        elif status(o) != "active":
            continue
        elif rt == "subtechnique-of" and src in tech:
            tech[src]["parent"] = dst
        elif rt == "mitigates" and src in mit and dst in tech:
            if mit[src]["status"] == "active" and tech[dst]["status"] == "active":
                mitigates.add((src, dst))
    return dict(version=v, tactics=ta_by_id, tech=tech, mit=mit, revoked_by=revoked_by, mitigates=mitigates)

snaps = {}
for v in VERSIONS:
    s = snaps[v] = load_snapshot(v)
    st = collections.Counter(t["status"] for t in s["tech"].values())
    act = [t for t in s["tech"].values() if t["status"] == "active"]
    print(f"v{v:5s} tactic={len(s['tactics']):2d}  kỹ thuật active={st['active']:4d} "
          f"(cha={sum(not t['is_sub'] for t in act):3d}, con={sum(t['is_sub'] for t in act):3d})  "
          f"revoked={st['revoked']:3d}  deprecated={st['deprecated']:3d}")

v6.3   tactic=12  kỹ thuật active= 266 (cha=266, con=  0)  revoked=  0  deprecated=  0
v7.2   tactic=12  kỹ thuật active= 428 (cha=156, con=272)  revoked=129  deprecated= 11
v8.2   tactic=14  kỹ thuật active= 530 (cha=178, con=352)  revoked=129  deprecated= 11
v9.0   tactic=14  kỹ thuật active= 552 (cha=185, con=367)  revoked=129  deprecated= 11
v10.1  tactic=14  kỹ thuật active= 566 (cha=188, con=378)  revoked=129  deprecated= 12
v11.3  tactic=14  kỹ thuật active= 576 (cha=191, con=385)  revoked=131  deprecated= 12
v12.1  tactic=14  kỹ thuật active= 594 (cha=193, con=401)  revoked=131  deprecated= 12
v13.1  tactic=14  kỹ thuật active= 607 (cha=196, con=411)  revoked=131  deprecated= 12
v14.1  tactic=14  kỹ thuật active= 625 (cha=201, con=424)  revoked=131  deprecated= 12
v15.1  tactic=14  kỹ thuật active= 637 (cha=202, con=435)  revoked=131  deprecated= 12
v16.1  tactic=14  kỹ thuật active= 656 (cha=203, con=453)  revoked=131  deprecated= 12
v17.1  tactic=14  kỹ thuật active= 679 (cha

## 4. So sánh hai phiên bản liên tiếp

Hàm `diff(A, B)` sinh ra danh sách **sự kiện thay đổi**. Mỗi sự kiện ghi kiểu thay đổi, mã cũ, mã mới (nếu có), tên trước/sau và chi tiết. Một kỹ thuật có thể sinh nhiều sự kiện trong cùng một lần chuyển, ví dụ vừa đổi tên vừa đổi tactic.

In [4]:
def resolve(snapshot, i, max_hops=10):
    # Theo chuỗi revoked-by trong CÙNG một phiên bản tới khi gặp đối tượng còn hiệu lực
    path = [i]
    while snapshot["tech"].get(i, {}).get("status") == "revoked" and i in snapshot["revoked_by"] and len(path) <= max_hops:
        i = snapshot["revoked_by"][i]
        path.append(i)
    return i, path

def diff(A, B):
    ev = []
    def add(kind, old, new=None, **kw):
        ev.append(dict(v_from=A["version"], v_to=B["version"], type=kind, id_old=old, id_new=new,
                       name_old=A["tech"].get(old, {}).get("name") if old else None,
                       name_new=B["tech"].get(new, {}).get("name") if new else None, **kw))
    ta, tb = A["tech"], B["tech"]
    act_a = {i for i, t in ta.items() if t["status"] == "active"}
    act_b = {i for i, t in tb.items() if t["status"] == "active"}

    for i in sorted(act_b - set(ta)):
        add("ADDED", None, i, detail="sub-technique" if tb[i]["is_sub"] else "technique")
    revoked_targets = collections.defaultdict(list)
    for i in sorted(act_a):
        sb = tb.get(i, {}).get("status")
        if sb == "revoked":
            tgt, path = resolve(B, i)
            kind = "REVOKED→EXISTING" if tgt in act_a else "REVOKED→NEW"
            add(kind, i, tgt, detail=" → ".join(path))
            revoked_targets[tgt].append(i)
        elif sb == "deprecated":
            add("DEPRECATED", i)
        elif sb is None:
            add("DISAPPEARED", i)   # không mong đợi: báo để kiểm tra
    for tgt, olds in revoked_targets.items():
        if len(olds) >= 2:
            add("MERGE", ",".join(olds), tgt, detail=f"{len(olds)} mã gộp vào {tgt}")

    for i in sorted(act_a & act_b):
        a, b = ta[i], tb[i]
        if a["name"] != b["name"]:
            add("RENAMED", i, i, detail=f"{a['name']} → {b['name']}")
        if a["tactics"] != b["tactics"]:
            add("TACTIC_CHANGED", i, i, detail=f"{list(a['tactics'])} → {list(b['tactics'])}")
        if a["parent"] != b["parent"]:
            add("REPARENTED", i, i, detail=f"{a['parent']} → {b['parent']}")
        ma = {m for m, t in A["mitigates"] if t == i}; mb = {m for m, t in B["mitigates"] if t == i}
        if ma != mb:
            add("MITIGATION_CHANGED", i, i, detail=f"+{sorted(mb - ma)} -{sorted(ma - mb)}")
        if int(str(b["version"]).split(".")[0]) > int(str(a["version"]).split(".")[0]):
            add("MAJOR_REVISION", i, i, detail=f"v{a['version']} → v{b['version']}")
    children_a = collections.Counter(t["parent"] for i, t in ta.items() if i in act_a and t["parent"])
    new_children = collections.Counter(tb[i]["parent"] for i in act_b - act_a if tb[i]["parent"])
    for p, n in sorted(new_children.items()):   # sắp xếp để thứ tự đầu ra tất định
        if p in act_a and p in act_b:
            add("REFINED", p, p, detail=f"+{n} sub-technique (trước có {children_a.get(p, 0)})")

    # Tactic
    for tid in sorted(set(A["tactics"]) | set(B["tactics"])):
        a, b = A["tactics"].get(tid), B["tactics"].get(tid)
        if a and not b:
            ev.append(dict(v_from=A["version"], v_to=B["version"], type="TACTIC_REMOVED", id_old=tid, name_old=a[1]))
        elif b and not a:
            ev.append(dict(v_from=A["version"], v_to=B["version"], type="TACTIC_ADDED", id_new=tid, name_new=b[1]))
        elif a != b:
            ev.append(dict(v_from=A["version"], v_to=B["version"], type="TACTIC_REDEFINED", id_old=tid, id_new=tid,
                           name_old=a[1], name_new=b[1], detail=f"{a[0]} → {b[0]}"))
    return ev

events = []
for a, b in zip(VERSIONS, VERSIONS[1:]):
    events += diff(snaps[a], snaps[b])
ev = pd.DataFrame(events)
ev["transition"] = ev.v_from + "→" + ev.v_to
ev.to_csv(OUT / "change_events.csv", index=False)
print(f"Tổng số sự kiện thay đổi: {len(ev)}")
assert not (ev.type == "DISAPPEARED").any(), "Có mã biến mất không rõ lý do — cần kiểm tra"

Tổng số sự kiện thay đổi: 1564


## 5. Tổng hợp theo từng lần chuyển phiên bản

Bảng dưới là nguồn cho **bảng/biểu đồ động cơ** trong bài: mỗi lần MITRE ra bản lớn, bao nhiêu mã bị thu hồi, đổi tên, đổi tactic... Hai lần chuyển đáng chú ý nhất là **6.3→7.2** (đưa vào sub-technique) và **18.1→19.0** (tách *Defense Evasion*).

In [5]:
ORDER = ["ADDED", "REVOKED→NEW", "REVOKED→EXISTING", "MERGE", "DEPRECATED", "RENAMED", "TACTIC_CHANGED",
         "REPARENTED", "MITIGATION_CHANGED", "MAJOR_REVISION", "REFINED",
         "TACTIC_ADDED", "TACTIC_REDEFINED", "TACTIC_REMOVED"]
summary = ev.pivot_table(index="transition", columns="type", values="v_to", aggfunc="count", fill_value=0)
summary = summary.reindex(columns=[c for c in ORDER if c in summary.columns])
summary = summary.reindex([f"{a}→{b}" for a, b in zip(VERSIONS, VERSIONS[1:])]).fillna(0).astype(int)
summary.to_csv(OUT / "summary_by_transition.csv")
summary

type,ADDED,REVOKED→NEW,REVOKED→EXISTING,MERGE,DEPRECATED,RENAMED,TACTIC_CHANGED,MITIGATION_CHANGED,MAJOR_REVISION,REFINED,TACTIC_ADDED,TACTIC_REDEFINED
transition,,,,,,,,,,,,
6.3→7.2,302,126,3,5,11,17,10,10,17,43,0,0
7.2→8.2,102,0,0,0,0,4,0,5,12,12,2,0
8.2→9.0,22,0,0,0,0,5,5,14,13,7,0,0
9.0→10.1,15,0,0,0,1,2,0,44,2,9,0,0
10.1→11.3,12,1,1,0,0,9,0,20,23,9,0,0
11.3→12.1,18,0,0,0,0,2,0,8,2,11,0,0
12.1→13.1,13,0,0,0,0,1,0,15,1,9,0,0
13.1→14.1,18,0,0,0,0,1,6,14,2,13,0,0
14.1→15.1,12,0,0,0,0,3,0,24,5,11,0,0


In [6]:
# Mức "phá vỡ" mỗi lần chuyển: số mã đang dùng bị mất hiệu lực hoặc đổi nghĩa
BREAKING = ["REVOKED→NEW", "REVOKED→EXISTING", "DEPRECATED", "TACTIC_CHANGED", "REPARENTED", "TACTIC_REDEFINED"]
br = summary.reindex(columns=[c for c in BREAKING if c in summary.columns]).sum(axis=1).rename("breaking_changes")
act_from = pd.Series({f"{a}→{b}": sum(t["status"] == "active" for t in snaps[a]["tech"].values())
                      for a, b in zip(VERSIONS, VERSIONS[1:])}, name="active_before")
tbl = pd.concat([act_from, br], axis=1)
tbl["pct_affected"] = (100 * tbl.breaking_changes / tbl.active_before).round(1)
tbl

,active_before,breaking_changes,pct_affected
6.3→7.2,266,150,56.4
7.2→8.2,428,0,0.0
8.2→9.0,530,5,0.9
9.0→10.1,552,1,0.2
10.1→11.3,566,2,0.4
11.3→12.1,576,0,0.0
12.1→13.1,594,0,0.0
13.1→14.1,607,6,1.0
14.1→15.1,625,0,0.0
15.1→16.1,637,0,0.0


## 6. Đối chiếu với release notes chính thức của MITRE cho v19

Đây là bước **kiểm định**: số liệu notebook tự tính phải khớp với công bố của MITRE. Theo release notes v19 (28/4/2026) mà bản rà soát tài liệu đã ghi lại:
- Enterprise từ **14 tactic / 216 kỹ thuật** lên **15 tactic / 222 kỹ thuật** (kỹ thuật ở đây là kỹ thuật cha, không tính sub-technique).
- **17** kỹ thuật Enterprise bị thu hồi.
- Clear Windows Event Logs chuyển sang **T1685.005**.

Nếu có chênh lệch, cần tìm hiểu nguyên nhân (cách MITRE đếm khác cách mình đếm) và ghi rõ trong bài. **Không chỉnh code cho khớp số.**

In [7]:
if "18.1" in snaps and "19.0" in snaps:
    a, b = snaps["18.1"], snaps["19.0"]
    parents = lambda s: sum(1 for t in s["tech"].values() if t["status"] == "active" and not t["is_sub"])
    e19 = ev[ev.transition == "18.1→19.0"]
    rows = [
        ("Số tactic v18.1 → v19.0", "14 → 15", f"{len(a['tactics'])} → {len(b['tactics'])}"),
        ("Số kỹ thuật cha v18.1 → v19.0", "216 → 222", f"{parents(a)} → {parents(b)}"),
        ("Số kỹ thuật bị thu hồi (cha + con)", "17", str(int(e19.type.str.startswith("REVOKED").sum()))),
        ("Số kỹ thuật sửa lớn (tăng major version)", "khoảng 199 (theo ghi chú rà soát)",
         str(int((e19.type == "MAJOR_REVISION").sum()))),
        ("T1070.001 ở v19.0", "→ T1685.005", f"→ {resolve(b, 'T1070.001')[0]}"),
        ("TA0005 ở v19.0", "Stealth", b["tactics"]["TA0005"][1]),
    ]
    display(pd.DataFrame(rows, columns=["Chỉ số", "Release notes MITRE", "Notebook tính được"]))
    # Ca "cùng mã, đổi nghĩa": kỹ thuật vẫn nằm dưới TA0005 nhưng TA0005 giờ mang nghĩa khác
    stay = [i for i, t in b["tech"].items() if t["status"] == "active" and "TA0005" in t["tactics"]
            and a["tech"].get(i, {}).get("status") == "active" and "TA0005" in a["tech"][i]["tactics"]]
    moved = [i for i, t in b["tech"].items() if t["status"] == "active" and "TA0112" in t["tactics"]]
    print(f"Kỹ thuật vẫn gắn TA0005 (mã giữ nguyên, nghĩa tactic đổi): {len(stay)}")
    print(f"Kỹ thuật thuộc tactic mới TA0112 Defense Impairment ở v19.0: {len(moved)}")

,Chỉ số,Release notes MITRE,Notebook tính được
0,Số tactic v18.1 → v19.0,14 → 15,14 → 15
1,Số kỹ thuật cha v18.1 → v19.0,216 → 222,216 → 222
2,Số kỹ thuật bị thu hồi (cha + con),17,17
3,Số kỹ thuật sửa lớn (tăng major version),khoảng 199 (theo ghi chú rà soát),199
4,T1070.001 ở v19.0,→ T1685.005,→ T1685.005
5,TA0005 ở v19.0,Stealth,Stealth


Kỹ thuật vẫn gắn TA0005 (mã giữ nguyên, nghĩa tactic đổi): 144
Kỹ thuật thuộc tactic mới TA0112 Defense Impairment ở v19.0: 56


## 7. Bảng tra mã: mọi mã từng tồn tại → tình trạng ở bản mới nhất

Bảng này trả lời câu hỏi: *"Một tài liệu cũ ghi mã X, theo chuẩn hiện hành thì X là gì?"*

- `VALID`: còn hiệu lực, có thể đã đổi tên.
- `REVOKED`: đã thu hồi, kèm mã thay thế hiện hành (theo chuỗi `revoked-by`).
- `DEPRECATED`: ngừng dùng, **không có** mã thay thế.

Bảng này chính là **baseline "tra bảng ánh xạ"** mà phương pháp ở Trụ 3 phải vượt qua. Nó xử lý tốt đổi mã 1:1, nhưng **không** xử lý được tách 1:N, các thay đổi quan hệ, hay ca *cùng mã, đổi nghĩa* như TA0005.

In [8]:
latest = snaps[VERSIONS[-1]]
first_seen, last_active = {}, {}
for v in VERSIONS:
    for i, t in snaps[v]["tech"].items():
        first_seen.setdefault(i, v)
        if t["status"] == "active":
            last_active[i] = v
rows = []
for i in sorted(first_seen, key=lambda x: (x[:5], x)):
    t = latest["tech"].get(i)
    st = t["status"] if t else "absent"
    cur, path = resolve(latest, i) if st == "revoked" else (i if st == "active" else None, [i])
    rows.append(dict(id=i, name_first=snaps[first_seen[i]]["tech"][i]["name"], first_seen=first_seen[i],
                     last_active=last_active.get(i), status_latest={"active": "VALID", "revoked": "REVOKED",
                     "deprecated": "DEPRECATED"}.get(st, st.upper()),
                     current_id=cur, current_name=latest["tech"].get(cur, {}).get("name") if cur else None,
                     chain=" → ".join(path)))
res = pd.DataFrame(rows)
res.to_csv(OUT / "id_resolution_latest.csv", index=False)
print(res.status_latest.value_counts().to_string())
print(f"\nTrong {len(res)} mã kỹ thuật từng xuất hiện ở các phiên bản đã phân tích, "
      f"{(res.status_latest != 'VALID').mean():.1%} không còn hiệu lực ở v{VERSIONS[-1]}.")
res[res.status_latest != "VALID"].sample(8, random_state=1)

status_latest
VALID         697
REVOKED       149
DEPRECATED     12

Trong 858 mã kỹ thuật từng xuất hiện ở các phiên bản đã phân tích, 18.8% không còn hiệu lực ở v19.2.


,id,name_first,first_seen,last_active,status_latest,current_id,current_name,chain
206,T1085,Rundll32,6.3,6.3,REVOKED,T1218.011,Rundll32,T1085 → T1218.011
214,T1089,Disabling Security Tools,6.3,6.3,REVOKED,T1685,Disable or Modify Tools,T1089 → T1685
349,T1182,AppCert DLLs,6.3,6.3,REVOKED,T1546.009,AppCert DLLs,T1182 → T1546.009
313,T1148,HISTCONTROL,6.3,6.3,REVOKED,T1690,Prevent Command History Logging,T1148 → T1690
675,T1574.002,DLL Side-Loading,7.2,16.1,REVOKED,T1574.001,DLL,T1574.002 → T1574.001
68,T1028,Windows Remote Management,6.3,6.3,REVOKED,T1021.006,Windows Remote Management,T1028 → T1021.006
359,T1192,Spearphishing Link,6.3,6.3,REVOKED,T1566.002,Spearphishing Link,T1192 → T1566.002
171,T1070.001,Clear Windows Event Logs,7.2,18.1,REVOKED,T1685.005,Clear Windows Event Logs,T1070.001 → T1685.005


## 8. Danh sách thực thể đã đổi giữa v18.1 và bản mới nhất (đầu vào cho Trụ 2)

Đây là tập "trước/sau" dùng để sinh câu hỏi cho nghiên cứu có kiểm soát ở Trụ 2. Mỗi dòng có kiểu thay đổi, để kết quả phân tích được theo từng kiểu. Ở bước sau, em chạy notebook 02 với v18.1 và với bản mới nhất, rồi notebook 03 lọc câu hỏi theo các mã trong danh sách này.

In [9]:
if "18.1" in VERSIONS:
    seg = VERSIONS[VERSIONS.index("18.1"):]
    sub = ev[ev.v_from.isin(seg[:-1]) & ev.v_to.isin(seg[1:])]
    keep = ["REVOKED→NEW", "REVOKED→EXISTING", "MERGE", "DEPRECATED", "RENAMED", "TACTIC_CHANGED",
            "REPARENTED", "MITIGATION_CHANGED", "TACTIC_REDEFINED", "REFINED"]
    changed = sub[sub.type.isin(keep)].copy()
    changed.to_csv(OUT / f"changed_items_18.1_to_{VERSIONS[-1]}.csv", index=False)
    display(changed.groupby(["transition", "type"]).size().unstack(fill_value=0))

type,MERGE,REFINED,RENAMED,REVOKED→NEW,TACTIC_CHANGED,TACTIC_REDEFINED
transition,,,,,,
18.1→19.0,1,1,4,17,72,1


## 9. Ghi chú cho bài báo và việc tiếp theo

**Những gì notebook này cho phép em khẳng định** (sau khi chạy và kiểm tra lại):
- Mỗi bản lớn của ATT&CK làm mất hiệu lực hoặc thay đổi một tỷ lệ mã nhất định (bảng mục 5). Đây là số liệu động cơ định lượng.
- Có bao nhiêu mã từng tồn tại nay không còn hiệu lực (mục 7). Mọi tài liệu, luật phát hiện hay mô hình học từ dữ liệu cũ đều có nguy cơ dùng các mã này.
- v19 có ca **cùng mã, đổi nghĩa** (TA0005), là ca mà kiểm tra "mã có tồn tại không" sẽ bỏ sót.

**Giới hạn phải ghi trong bài:**
- *Tách 1→N* không suy ra trọn vẹn được từ STIX. `REFINED` và `DEPRECATED` chỉ là dấu hiệu, cần đối chiếu release notes.
- `MAJOR_REVISION` là *ứng viên* đổi nghĩa (MITRE tăng phiên bản chính khi sửa lớn). Muốn khẳng định là đổi nghĩa thì cần kiểm tra thủ công một mẫu.
- Chỉ phân tích ATT&CK Enterprise. Mobile và ICS có thể mở rộng ở bản tạp chí.

**Việc tiếp theo:**
1. Chạy notebook, kiểm tra bảng đối chiếu ở mục 6, và đọc 20 sự kiện ngẫu nhiên trong `change_events.csv` xem có hợp lý không.
2. Đọc release notes v19.0–19.2 để lập danh sách *tách* đầy đủ, bổ sung bằng tay vào một file `splits_manual.csv`.
3. Chạy notebook `02_benchmark_theo_phien_ban` hai lần (v19.2 và v18.1) để có bộ câu hỏi trước/sau cho pilot Trụ 2.